# RQ3: Dynamics — Temporal Maturity Transitions

*How do repositories transition between maturity levels over time, and where does progression stall?*

RQ1 established that AI artifact adoption follows a cumulative maturity progression (Guttman CR = 0.997 on the strict+W+ population: 210 repos scored from the file-level predictions dataset, with the 231 whitelist-excluded repos padded as L1 on the full 441-repo frame). Note that RQ3 analyzes only repos with artifact timelines \u2014 padded L1 repos have no artifacts and hence no timelines, so they do not enter this notebook. RQ2 confirmed this structure is robust. RQ3 completes the narrative arc by examining **temporal dynamics**: when categories are introduced, how artifacts are maintained, and when they are abandoned.

> **Attribution note**: timeline categories come from the **validated notebook-13 predictor** (`data/rq1_file_predictions.parquet`, `assigned_category`) — the same attribution behind `rq1_repo_scores.csv` in RQ1/RQ2. Under this attribution the strict+W+ corpus contains **no `flows`/`session-logs` (L4-category) files**, so the timelines contain no L4 events — fully consistent with RQ1's finding that no repository certifies L4. (An earlier version of this notebook joined categories from `artifacts_all.csv`, whose stale pre-validation labeling produced 40 spurious L4-category introduction events; those numbers are superseded.)

| Sub-question | Focus | Key method |
|---|---|---|
| **RQ3a** | Do repos follow sequential level progression? | Kaplan-Meier survival, sequence analysis |
| **RQ3b** | How common are reversals, plateaus, and abandonment? | Lifecycle classification, chi-squared |
| **RQ3c** | Does maintenance effort scale with maturity level? | Kruskal-Wallis, partial Spearman |

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OMP_MAX_ACTIVE_LEVELS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import sys
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy.stats import chi2_contingency, kruskal, mannwhitneyu, spearmanr

pio.renderers.default = "notebook"
warnings.filterwarnings("ignore")

try:
    from lifelines import KaplanMeierFitter
    from lifelines.statistics import logrank_test
    HAS_LIFELINES = True
except ImportError:
    HAS_LIFELINES = False
    print("WARNING: lifelines not installed. Kaplan-Meier plots will use matplotlib fallback.")

PROJECT_ROOT = Path("../..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.maturity_scorer import CATEGORY_NAMES, CATEGORY_TO_LEVEL, MATURITY_LABELS, MaturityLevel
from src.temporal_health import classify_artifact_lifecycle

print(f"Project root: {PROJECT_ROOT}")
print(f"lifelines available: {HAS_LIFELINES}")

In [ ]:
# --- Paths ---
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "output"
FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(exist_ok=True)

# --- Constants ---
LEVEL_COLORS = {1: "#6b7280", 2: "#3b82f6", 3: "#f97316", 4: "#22c55e"}
LEVEL_LABELS = {
    1: "L1 Ad Hoc",
    2: "L2 Grounded",
    3: "L3 Agent-Augmented",
    4: "L4-category (Orchestration)",
}

CATEGORY_TO_LEVEL_INT = {
    cat: int(lvl) for cat, lvl in CATEGORY_TO_LEVEL.items()
}

LEVEL_TO_CATEGORIES = {}
for cat, lvl in CATEGORY_TO_LEVEL_INT.items():
    LEVEL_TO_CATEGORIES.setdefault(lvl, []).append(cat)

def save_fig(fig, filename, width=1200, height=600):
    stem = Path(filename).stem
    for ext, kwargs in ((".png", {"scale": 2}), (".pdf", {})):
        path = FIGURES_DIR / f"{stem}{ext}"
        try:
            fig.write_image(str(path), width=width, height=height, **kwargs)
            print(f"Saved: {path}")
        except Exception as e:
            print(f"Could not save {path}: {e}")

def cramers_v(contingency_table):
    """Compute Cramer's V from a contingency table."""
    chi2 = chi2_contingency(contingency_table)[0]
    n = contingency_table.sum().sum()
    min_dim = min(contingency_table.shape) - 1
    return np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else 0.0

print("Config loaded.")

In [ ]:
# --- Load base data ---
# Strict AI-artifact filter (same WL_STEPS definition as notebooks 13/14),
# extended with the W+ exact-basename recoveries (nested CLAUDE.md/AGENTS.md,
# mcp configs, ...) to match RQ1/RQ2.
# rq1_repo_scores.csv is already strict+W+ (rebuilt by RQ1 with STRICT_MODE + WL_PLUS).
# artifacts_all.csv has no discovery_step column, so strict membership is
# resolved via (repo_name, artifact_path) pairs from filtered_common_metadata;
# the cell-6 inner join then restricts all timeline events to strict+W+ artifacts.
STRICT_MODE = True
WL_STEPS = {"tool_standard", "shared_in_tool_folder", "shared_in_root"}
WL_PLUS = True

from src.artifact_filtering import load_protected_patterns
PROTECTED_EXACT, _, _ = load_protected_patterns(str(PROJECT_ROOT / "Artifacts"))

def wl_mask(df, name_col="artifact_name"):
    m = df["discovery_step"].isin(WL_STEPS)
    if WL_PLUS:
        m |= df[name_col].str.lower().isin(PROTECTED_EXACT)
    return m

repo_scores = pd.read_csv(DATA_DIR / "rq1_repo_scores.csv")
artifacts_all = pd.read_csv(DATA_DIR / "artifacts_all.csv")
ai_tools_meta = pd.read_csv(DATA_DIR / "filtered_ai_tools_metadata.csv")

if STRICT_MODE:
    common_meta = pd.read_csv(DATA_DIR / "filtered_common_metadata.csv")
    strict_meta = common_meta[wl_mask(common_meta)]
    strict_pairs = set(zip(strict_meta["repo_name"], strict_meta["artifact_path"]))
    artifacts_all = artifacts_all[
        [pair in strict_pairs for pair in zip(artifacts_all["repo_name"], artifacts_all["artifact_path"])]
    ].reset_index(drop=True)
    ai_tools_meta = ai_tools_meta[wl_mask(ai_tools_meta)].reset_index(drop=True)

ai_tool_repos = set(ai_tools_meta["repo_name"].unique())

print(f"Repo scores: {len(repo_scores)} repos")
print(f"Artifacts: {len(artifacts_all)} total")
print(f"AI-tools subset: {len(ai_tool_repos)} repos")
print(f"Level distribution:\n{repo_scores['level'].value_counts().sort_index().to_string()}")

---
## Section 1: Data Assembly

Assemble temporal data from per-repo timeseries CSVs. Join with category assignments to build per-repo category and level introduction timelines.

In [ ]:
# Glob and concatenate all artifact timeseries CSVs
CACHE_TIMELINES = DATA_DIR / "rq3_timelines.csv"

if CACHE_TIMELINES.exists():
    print(f"Loading cached timelines from {CACHE_TIMELINES}")
    timeseries_all = pd.read_csv(CACHE_TIMELINES, parse_dates=["commit_date"])
else:
    ts_files = sorted(OUTPUT_DIR.glob("**/*_artifact_timeseries.csv"))
    print(f"Found {len(ts_files)} timeseries files")

    frames = []
    for f in ts_files:
        try:
            df = pd.read_csv(f)
            # Extract repo_name from path: output/org/repo/...
            parts = f.relative_to(OUTPUT_DIR).parts
            if len(parts) >= 2:
                repo_name = f"{parts[0]}/{parts[1]}"
            else:
                repo_name = parts[0]
            df["repo_name"] = repo_name
            frames.append(df)
        except Exception as e:
            pass

    timeseries_all = pd.concat(frames, ignore_index=True)
    timeseries_all["commit_date"] = pd.to_datetime(timeseries_all["commit_date"], utc=True)
    timeseries_all.to_csv(CACHE_TIMELINES, index=False)
    print(f"Cached timelines to {CACHE_TIMELINES}")

print(f"Total timeseries events: {len(timeseries_all):,}")
print(f"Repos with timeseries: {timeseries_all['repo_name'].nunique()}")
print(f"Date range: {timeseries_all['commit_date'].min()} — {timeseries_all['commit_date'].max()}")
print(f"Actions: {timeseries_all['action'].value_counts().to_dict()}")

In [ ]:
# Join timeseries with per-file category assignments from the VALIDATED
# notebook-13 predictor (data/rq1_file_predictions.parquet, assigned_category)
# — the same attribution that produces rq1_repo_scores.csv for RQ1/RQ2.
# NOTE: earlier versions of this notebook joined categories from
# artifacts_all.csv, whose older categorization disagrees with the validated
# predictor — notably labeling ~100 strict files as flows/session-logs (L4)
# that the validated pipeline assigns to L2/L3 categories or none. Under the
# validated attribution the strict+W+ corpus contains no flows/session-logs
# files, so timelines contain no L4-category events.
pred_df = pd.read_parquet(DATA_DIR / "rq1_file_predictions.parquet").rename(columns={"repo": "repo_name"})
pred_df = pred_df.dropna(subset=["assigned_category"])
pred_df = pred_df[pred_df["assigned_category"].isin(CATEGORY_TO_LEVEL_INT)]
pred_df = pred_df[[pair in strict_pairs for pair in zip(pred_df["repo_name"], pred_df["artifact_path"])]]

artifact_categories = (
    pred_df[["repo_name", "artifact_path", "assigned_category"]]
    .rename(columns={"assigned_category": "category"})
    .drop_duplicates()
)

timeseries_with_cat = timeseries_all.merge(
    artifact_categories,
    on=["repo_name", "artifact_path"],
    how="inner"
)

# Add level from category
timeseries_with_cat["level"] = timeseries_with_cat["category"].map(CATEGORY_TO_LEVEL_INT)

print(f"Timeseries events with category: {len(timeseries_with_cat):,} / {len(timeseries_all):,} ({100*len(timeseries_with_cat)/len(timeseries_all):.1f}%)")
print(f"Repos with categorized timeseries: {timeseries_with_cat['repo_name'].nunique()}")
print(f"Categories represented: {sorted(timeseries_with_cat['category'].unique())}")


In [ ]:
# Build per-repo category introduction timeline: first "created" event per (repo, category)
created_events = timeseries_with_cat[timeseries_with_cat["action"] == "created"].copy()

cat_intro = (
    created_events
    .groupby(["repo_name", "category"])["commit_date"]
    .min()
    .reset_index()
    .rename(columns={"commit_date": "intro_date"})
)
cat_intro["level"] = cat_intro["category"].map(CATEGORY_TO_LEVEL_INT)

# Build per-repo level introduction timeline: earliest category intro per level
level_intro = (
    cat_intro
    .groupby(["repo_name", "level"])["intro_date"]
    .min()
    .reset_index()
    .rename(columns={"intro_date": "level_intro_date"})
)

# Merge with repo scores for current maturity level
level_intro = level_intro.merge(
    repo_scores[["full_repo_name", "level"]].rename(
        columns={"full_repo_name": "repo_name", "level": "current_level"}
    ),
    on="repo_name",
    how="left"
)

print(f"Category introductions: {len(cat_intro):,} across {cat_intro['repo_name'].nunique()} repos")
print(f"Level introductions: {len(level_intro):,}")
print(f"\nCategory intro counts:")
print(cat_intro["category"].value_counts().to_string())

---
## Section 2: RQ3a — Sequential Progression

*Do repos follow sequential level progression (L2 → L3 → L4), or do they skip levels?*

We classify each repo's level introduction sequence and measure transition times. Sequences are written from the **L1 base**: L1 is the pre-adoption state (absence of strict artifacts), so every adopter's sequence starts there and its first hop **is** the L1 exit — e.g. `L1→L2` (adopt grounding and stop), `L1→L2→L3` (full sequential climb), `L1→L3` (direct jump). The `pattern` taxonomy below classifies the *post-entry* trajectory: 'single-level' = one hop out of L1 and no further level introductions.

**Note on "L4"**: under the validated attribution there are no L4-category (`flows`/`session-logs`) artifacts in the strict+W+ corpus, so no L4 introductions occur; observed sequences span L2–L3 only. See the attribution note in the notebook header.

In [ ]:
# Classify level introduction sequences per repo
def classify_progression(repo_levels_df):
    """Classify a repo's level intro sequence.
    
    Returns: (pattern, sequence_str)
    - 'sequential': L2→L3→L4 in order
    - 'forward-with-skip': skips a level (e.g., L2→L4)
    - 'non-sequential': out of order (e.g., L3→L2)
    - 'single-level': only one level introduced
    """
    sorted_df = repo_levels_df.sort_values("level_intro_date")
    levels = sorted_df["level"].tolist()
    # Every adopter starts at L1 (absence of artifacts), so sequences are
    # written from the L1 base; the first hop is the entry transition (3a.6).
    seq_str = "L1→" + "→".join(f"L{l}" for l in levels)
    
    if len(levels) == 1:
        return "single-level", seq_str
    
    # Check if strictly increasing
    is_increasing = all(levels[i] < levels[i+1] for i in range(len(levels)-1))
    if not is_increasing:
        return "non-sequential", seq_str
    
    # Check for skips
    has_skip = any(levels[i+1] - levels[i] > 1 for i in range(len(levels)-1))
    if has_skip:
        return "forward-with-skip", seq_str
    
    return "sequential", seq_str

# Apply to each repo
repo_sequences = []
for repo, grp in level_intro.groupby("repo_name"):
    pattern, seq_str = classify_progression(grp)
    current_lvl = grp["current_level"].iloc[0] if grp["current_level"].notna().any() else None
    repo_sequences.append({
        "repo_name": repo,
        "pattern": pattern,
        "sequence": seq_str,
        "n_levels": len(grp),
        "current_level": current_lvl,
    })

seq_df = pd.DataFrame(repo_sequences)

print("Progression pattern distribution:")
print(seq_df["pattern"].value_counts().to_string())
print(f"\nTotal repos with level intros: {len(seq_df)}")

In [ ]:
# Chart 3a.1: Progression pattern distribution by current level (stacked bar)
pattern_order = ["sequential", "forward-with-skip", "non-sequential", "single-level"]
pattern_colors = {
    "sequential": "#22c55e",
    "forward-with-skip": "#f97316",
    "non-sequential": "#ef4444",
    "single-level": "#6b7280",
}

seq_with_level = seq_df.dropna(subset=["current_level"])
ct = pd.crosstab(seq_with_level["current_level"], seq_with_level["pattern"])

fig = go.Figure()
for pat in pattern_order:
    if pat in ct.columns:
        fig.add_trace(go.Bar(
            name=pat,
            x=[LEVEL_LABELS.get(int(l), f"L{int(l)}") for l in ct.index],
            y=ct[pat],
            marker_color=pattern_colors[pat],
        ))

fig.update_layout(
    title="RQ3a.1: Progression Pattern Distribution by Current Level",
    barmode="stack",
    xaxis_title="Current Maturity Level",
    yaxis_title="Number of Repos",
    legend_title="Pattern",
    template="plotly_white",
    width=700, height=450,
)
fig.show()
save_fig(fig, "rq3a_progression_patterns.png", width=700, height=450)

In [ ]:
# Chart 3a.2: Top 15 most common sequences (horizontal bar)
top_seqs = seq_df["sequence"].value_counts().head(15)

fig = go.Figure(go.Bar(
    y=top_seqs.index[::-1],
    x=top_seqs.values[::-1],
    orientation="h",
    marker_color="#3b82f6",
))
fig.update_layout(
    title="RQ3a.2: Top 15 Most Common Level Sequences (from L1 base)<br><sub>first hop = entry transition out of L1; see 3a.6 for entry timing</sub>",
    xaxis_title="Number of Repos",
    yaxis_title="Level Sequence",
    template="plotly_white",
    width=700, height=500,
    margin=dict(l=120),
)
fig.show()
save_fig(fig, "rq3a_top_sequences.png", width=700, height=500)

In [ ]:
# Transition times: days between level introductions
# Build pivot: repo_name → {level: intro_date}
level_pivot = level_intro.pivot_table(
    index="repo_name", columns="level", values="level_intro_date", aggfunc="first"
)

transitions = {}
for label, (l_from, l_to) in [("L2→L3", (2, 3)), ("L3→L4", (3, 4)), ("L2→L4", (2, 4))]:
    if l_from in level_pivot.columns and l_to in level_pivot.columns:
        mask = level_pivot[l_from].notna() & level_pivot[l_to].notna()
        days = (level_pivot.loc[mask, l_to] - level_pivot.loc[mask, l_from]).dt.total_seconds() / 86400
        # Only forward transitions (positive days)
        days = days[days > 0]
        transitions[label] = days

for label, days in transitions.items():
    print(f"{label}: n={len(days)}, median={days.median():.0f} days, mean={days.mean():.0f} days, IQR=[{days.quantile(0.25):.0f}, {days.quantile(0.75):.0f}]")

# Mann-Whitney U: L2→L3 vs L3→L4
if "L2→L3" in transitions and "L3→L4" in transitions and len(transitions["L2→L3"]) >= 5 and len(transitions["L3→L4"]) >= 5:
    u_stat, p_val = mannwhitneyu(transitions["L2→L3"], transitions["L3→L4"], alternative="two-sided")
    print(f"\nMann-Whitney U (L2→L3 vs L3→L4): U={u_stat:.0f}, p={p_val:.4f}")
else:
    print("\nInsufficient data for Mann-Whitney U test")

### RQ3a.6: Entry Transition — Leaving L1 (the first hop of every sequence above)

L1 is defined by the *absence* of strict artifacts, so it has no artifact events of its own: every adopter's first categorized introduction **is** its L1 exit. Two views:

- **Entry level** — does the repo enter the ladder at L2 (grounding first) or jump directly to L3?
- **L1 duration (adoption latency)** — how long did the repo demonstrably exist before its first strict artifact? The clock starts at the repo's earliest *observed* event in the full artifact-candidate timeseries (which tracks uncategorized files such as READMEs), so this is a **lower bound** on true time in L1: per-repo metrics carry only window-truncated clone dates, and repos whose observation begins at adoption register zero latency.

The entry latency joins the transition-time chart (3a.3) as the ladder's first rung. Chart 3a.4 additionally draws a **censored L1→L2 Kaplan-Meier curve** on the full 441-repo frame: the whitelist-excluded repos that RQ1 pads as L1 never introduce a strict artifact but do appear in the artifact-candidate timeseries, so they supply the right-censored mass that the adopters-only timeline frame lacks. The clock starts at the **observation-era dawn** — the corpus's first strict categorized introduction — or the repo's first observed candidate event, whichever is later, so time in L1 accrues only while adopting an AI-config artifact was actually possible (pre-era repo age would otherwise dominate the durations). The event is the first L2-category introduction; repos with no strict artifact are right-censored at the end of observation. The age-based clock (repo age at first L2 artifact) is retained as a secondary statistic.


In [ ]:
# --- Analysis 3a.6: Entry transition — leaving L1 (L1→L2 / L1→L3) ---
first_intro_any = created_events.groupby("repo_name")["commit_date"].min().rename("first_intro")
entry_level = (
    created_events.merge(first_intro_any, on="repo_name")
    .query("commit_date == first_intro")
    .groupby("repo_name")["level"].min()
)
print("Entry level (level of first categorized introduction):")
for lvl, n in entry_level.value_counts().sort_index().items():
    print(f"  L1\u2192L{int(lvl)}: {n} repos ({n/len(entry_level)*100:.1f}%)")

# L1 duration: earliest observed event in the FULL candidate timeseries -> first strict intro
first_observed = (
    timeseries_all[timeseries_all["repo_name"].isin(first_intro_any.index)]
    .groupby("repo_name")["commit_date"].min()
)
latency = ((first_intro_any - first_observed).dt.total_seconds() / 86400).rename("l1_days")

print(f"\nL1 duration (observed pre-adoption history \u2192 first strict artifact), n={len(latency)}:")
print(f"  median={latency.median():.0f}d, mean={latency.mean():.0f}d, IQR=[{latency.quantile(.25):.0f}, {latency.quantile(.75):.0f}]")
zero = (latency <= 0).sum()
print(f"  repos entering observation already with artifacts (zero latency): {zero} ({zero/len(latency)*100:.1f}%)")
pos = latency[latency > 0]
print(f"  among repos with measurable pre-adoption history (n={len(pos)}): median={pos.median():.0f}d, IQR=[{pos.quantile(.25):.0f}, {pos.quantile(.75):.0f}]")

# Speed comparison: L1 exit vs L2→L3 (lower-bound latencies bias L1 SHORT,
# so a significant "L1 slower" result is conservative)
if "L2→L3" in transitions and len(transitions["L2→L3"]) >= 5 and len(pos) >= 5:
    _u, _p = mannwhitneyu(pos, transitions["L2→L3"], alternative="two-sided")
    print(f"\nMann-Whitney U (L1 exit {len(pos)} vs L2\u2192L3 {len(transitions['L2\u2192L3'])}): "
          f"U={_u:.0f}, p={_p:.2e} (medians {pos.median():.0f}d vs {transitions['L2\u2192L3'].median():.0f}d)")

ldf = pd.DataFrame({"l1_days": latency, "entry": entry_level})
print("\nL1 duration by entry level:")
for lvl, g in ldf.groupby("entry"):
    print(f"  L1\u2192L{int(lvl)}: n={len(g)}, median={g['l1_days'].median():.0f}d")

fig = go.Figure()
for lvl, color in [(2, LEVEL_COLORS[2]), (3, LEVEL_COLORS[3])]:
    vals = ldf.loc[ldf["entry"] == lvl, "l1_days"]
    fig.add_trace(go.Histogram(x=vals, name=f"L1\u2192L{lvl} (n={len(vals)})", marker_color=color, opacity=0.75, nbinsx=40))
fig.update_layout(
    title="RQ3a.6: Time in L1 Before First Strict Artifact<br><sub>clock = earliest observed event in the artifact-candidate timeseries \u2014 a lower bound on true L1 duration</sub>",
    xaxis_title="Days from first observed activity to first categorized introduction",
    yaxis_title="Repos", barmode="overlay",
    template="plotly_white", width=800, height=450,
)
fig.show()
save_fig(fig, "rq3a_l1_adoption_latency.png", width=800, height=450)


In [ ]:
# Chart 3a.3: Transition time box plots
# Reconcile chart n vs sequence counts: only strictly positive gaps are timed
if set([2, 3]).issubset(level_pivot.columns):
    _d = (level_pivot[[2, 3]].dropna()[3] - level_pivot[[2, 3]].dropna()[2]).dt.total_seconds()
    _fwd, _tied, _inv = int((_d > 0).sum()), int((_d == 0).sum()), int((_d < 0).sum())
    recon_note = (f"{_fwd + _tied} L2\u2192L3-sequence repos = {_fwd} timed + {_tied} same-day (zero gap)<br>"
                  f"{_inv} inverted (L3\u2192L2) shown in the concordance analysis")
else:
    recon_note = "no multi-level repos"

fig = go.Figure()
# Ladder's first rung: L1 -> entry level (3a.6 latency; measurable repos only, lower bound)
_l1_pos = latency[latency > 0]
fig.add_trace(go.Box(
    y=_l1_pos,
    name=f"L1→entry (≥)",
    marker_color=LEVEL_COLORS[1],
    boxmean="sd",
))
for label, color in [("L2→L3", LEVEL_COLORS[3]), ("L3→L4", LEVEL_COLORS[4]), ("L2→L4", "#8b5cf6")]:
    if label in transitions and len(transitions[label]) > 0:
        fig.add_trace(go.Box(
            y=transitions[label],
            name=label,
            marker_color=color,
            boxmean="sd",
        ))

fig.update_layout(
    title=(f"RQ3a.3: Transition Times Along the Ladder<br>"
           f"<sub>L1→entry: lower bound, observed pre-adoption history (n={len(_l1_pos)}; zero-latency excluded)<br>"
           f"{recon_note}</sub>"),
    yaxis_title="Days",
    template="plotly_white",
    width=760, height=480,
    showlegend=False,
)
fig.show()
save_fig(fig, "rq3a_transition_times.png", width=760, height=480)

In [ ]:
# Chart 3a.4: Kaplan-Meier survival curves — time-to-next-level with censoring
# For repos at L2: time to reach L3 (censored if never reached)
# For repos at L3: time to first L4-category artifact — none occur under the validated attribution, so the L3 curve is fully censored

max_date = timeseries_with_cat["commit_date"].max()

def build_km_data(from_level, to_level):
    """Build survival data for transition from_level -> to_level."""
    # Repos that have from_level intro
    repos_with_from = level_pivot[level_pivot[from_level].notna()].index
    
    durations = []
    events = []  # 1=transitioned, 0=censored
    for repo in repos_with_from:
        from_date = level_pivot.loc[repo, from_level]
        if to_level in level_pivot.columns and pd.notna(level_pivot.loc[repo, to_level]):
            to_date = level_pivot.loc[repo, to_level]
            days = (to_date - from_date).total_seconds() / 86400
            if days > 0:
                durations.append(days)
                events.append(1)
            else:
                # Non-forward: censor at max observation
                durations.append((max_date - from_date).total_seconds() / 86400)
                events.append(0)
        else:
            # Never reached to_level: right-censored
            durations.append((max_date - from_date).total_seconds() / 86400)
            events.append(0)
    
    return np.array(durations), np.array(events)

# L1→L2 with genuine censoring: population = the full 441-repo frame
# (strict+W+ scored repos + whitelist-excluded repos padded as L1 in RQ1),
# restricted to repos observed in the candidate timeseries. The padded-L1
# repos never introduce a strict artifact but do carry candidate timelines
# (READMEs, non-standard markdown), so they provide the right-censored mass
# that the adopters-only frame lacks. Clock = first observed candidate event
# (lower bound on time in L1, as in 3a.6); event = first L2-category intro.
frame_repos = set(common_meta["repo_name"].unique())
obs_start = (
    timeseries_all[timeseries_all["repo_name"].isin(frame_repos)]
    .groupby("repo_name")["commit_date"].min()
)
first_l2 = created_events[created_events["level"] == 2].groupby("repo_name")["commit_date"].min()

# Era clock: time in L1 accrues only once adopting was possible. The era dawns
# at the corpus's first strict categorized introduction; each repo's clock
# starts at max(its first observed candidate event, era dawn), so pre-era repo
# age (when no AI-config artifact existed to adopt) does not inflate durations.
# The age-based clock (first observed event) is kept as a secondary statistic.
ERA_START = created_events["commit_date"].min().normalize()
print(f"Observation era: {ERA_START.date()} → {max_date.date()} ({(max_date - ERA_START).days} days)")

def l1l2_km_arrays(clock_start_fn):
    dur, evt = [], []
    for _repo, _obs in obs_start.items():
        _start = clock_start_fn(_obs)
        if _repo in first_l2.index:
            dur.append(max((first_l2[_repo] - _start).total_seconds() / 86400, 0.0))
            evt.append(1)
        else:
            dur.append((max_date - _start).total_seconds() / 86400)
            evt.append(0)
    return np.array(dur), np.array(evt)

l1l2_durations, l1l2_events = l1l2_km_arrays(lambda obs: max(obs, ERA_START))  # era clock (plotted)
l1l2_age_dur, l1l2_age_evt = l1l2_km_arrays(lambda obs: obs)  # age clock (secondary)
print(f"L1→L2 KM population: {len(l1l2_durations)} frame repos observed in the candidate timeseries "
      f"({int(l1l2_events.sum())} adopt an L2 artifact, {int((1 - l1l2_events).sum())} right-censored; "
      f"{len(frame_repos) - len(l1l2_durations)} frame repos have no timeseries)")
if HAS_LIFELINES:
    _kmf_age = KaplanMeierFitter().fit(l1l2_age_dur, event_observed=l1l2_age_evt)
    print(f"Secondary (age clock): KM median repo age at first L2 artifact = {_kmf_age.median_survival_time_:.0f}d")

# Batch-adoption caveat: first-L2 events are not repo-independent when an org
# rolls out AI config across its repos in one commit wave.
_batch = first_l2.dt.date.value_counts()
if _batch.iloc[0] >= 10:
    _bd = _batch.index[0]
    _borgs = pd.Series([r.split("/")[0] for r in first_l2[first_l2.dt.date == _bd].index]).value_counts()
    print(f"NOTE: {_batch.iloc[0]} repos share the same first-L2 date ({_bd}), "
          f"{_borgs.iloc[0]} of them from org '{_borgs.index[0]}' — an org-level rollout; "
          f"L1→L2 events are clustered by org, not repo-independent")

fig = go.Figure()
km_results = {}

for label, from_l, to_l, color in [("L1→L2", None, None, LEVEL_COLORS[2]), ("L2→L3", 2, 3, LEVEL_COLORS[3]), ("L3→L4", 3, 4, LEVEL_COLORS[4])]:
    if from_l is None:
        durations, events = l1l2_durations, l1l2_events
    else:
        if from_l not in level_pivot.columns:
            continue
        durations, events = build_km_data(from_l, to_l)
    if len(durations) < 5:
        continue
    
    km_results[label] = (durations, events)
    
    if HAS_LIFELINES:
        kmf = KaplanMeierFitter()
        kmf.fit(durations, event_observed=events, label=label)
        print(f"{label}: n={len(durations)}, events={int(events.sum())}, KM median={kmf.median_survival_time_:.0f}d, "
              f"25% transitioned by t={kmf.percentile(0.75):.0f}d")
        timeline = kmf.survival_function_
        fig.add_trace(go.Scatter(
            x=timeline.index,
            y=timeline.iloc[:, 0],
            name=f"{label} (n={len(durations)}, events={events.sum()})",
            line=dict(color=color, width=2),
        ))
    else:
        # Matplotlib fallback: simple step function
        sorted_idx = np.argsort(durations)
        sorted_d = durations[sorted_idx]
        sorted_e = events[sorted_idx]
        n = len(sorted_d)
        surv = np.ones(n + 1)
        times = np.zeros(n + 1)
        for i in range(n):
            times[i + 1] = sorted_d[i]
            if sorted_e[i] == 1:
                surv[i + 1] = surv[i] * (1 - 1 / (n - i))
            else:
                surv[i + 1] = surv[i]
        fig.add_trace(go.Scatter(x=times, y=surv, name=label, line=dict(color=color, width=2)))

# Log-rank tests
if HAS_LIFELINES and "L1→L2" in km_results and "L2→L3" in km_results:
    d0, e0 = km_results["L1→L2"]
    d1, e1 = km_results["L2→L3"]
    lr01 = logrank_test(d0, d1, event_observed_A=e0, event_observed_B=e1)
    print(f"Log-rank L1→L2 vs L2→L3: statistic={lr01.test_statistic:.3f}, p={lr01.p_value:.2e}")

if HAS_LIFELINES and "L2→L3" in km_results and "L3→L4" in km_results:
    d1, e1 = km_results["L2→L3"]
    d2, e2 = km_results["L3→L4"]
    lr = logrank_test(d1, d2, event_observed_A=e1, event_observed_B=e2)
    print(f"Log-rank test: statistic={lr.test_statistic:.3f}, p={lr.p_value:.4f}")

fig.update_layout(
    title="RQ3a.4: Kaplan-Meier Survival — Time to Next Level Introduction<br><sub>categories from the validated notebook-13 attribution; no L4-category events exist, so no repo transitions beyond L3<br>L1→L2: full 441 frame, padded-L1 right-censored; era clock (dawn = first strict intro in corpus)</sub>",
    xaxis_title="Days Since Level Introduction",
    yaxis_title="Probability of Not Yet Transitioning",
    template="plotly_white",
    width=800, height=500,
)
fig.show()
save_fig(fig, "rq3a_kaplan_meier.png", width=800, height=500)

In [ ]:
# Chart 3a.5: Category adoption order within levels
# For each level, which category tends to be introduced first?
cat_intro_with_level = cat_intro.copy()

# Within each (repo, level), rank categories by intro date
cat_intro_with_level["rank_in_level"] = (
    cat_intro_with_level
    .groupby(["repo_name", "level"])["intro_date"]
    .rank(method="first")
)

# For repos with multiple categories at same level, what's the first?
first_cat_per_level = cat_intro_with_level[cat_intro_with_level["rank_in_level"] == 1]
first_counts = first_cat_per_level.groupby(["level", "category"]).size().reset_index(name="count")

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=["L2 First Category", "L3 First Category", "L4-category First"],
)

for idx, lvl in enumerate([2, 3, 4], 1):
    lvl_data = first_counts[first_counts["level"] == lvl].sort_values("count", ascending=True)
    if len(lvl_data) > 0:
        fig.add_trace(go.Bar(
            y=lvl_data["category"],
            x=lvl_data["count"],
            orientation="h",
            marker_color=LEVEL_COLORS[lvl],
            showlegend=False,
        ), row=1, col=idx)

fig.update_layout(
    title="RQ3a.5: First Category Adopted per Level",
    template="plotly_white",
    width=1000, height=400,
)
fig.show()
save_fig(fig, "rq3a_first_category.png", width=1000, height=400)

In [ ]:
# Robustness: AI-tools subset — chi-squared on pattern proportions
ai_seq = seq_df[seq_df["repo_name"].isin(ai_tool_repos)]

print("=== AI-Tools Subset Robustness ===")
print(f"Repos with sequences in AI-tools subset: {len(ai_seq)}")
print(f"\nPattern distribution (AI-tools):")
print(ai_seq["pattern"].value_counts().to_string())

# Chi-squared: compare pattern distributions full vs AI-tools
if len(ai_seq) >= 10:
    all_patterns = seq_df["pattern"].value_counts()
    ai_patterns = ai_seq["pattern"].value_counts()
    common_patterns = sorted(set(all_patterns.index) & set(ai_patterns.index))
    if len(common_patterns) >= 2:
        ct_robust = pd.DataFrame({
            "full": [all_patterns.get(p, 0) for p in common_patterns],
            "ai_tools": [ai_patterns.get(p, 0) for p in common_patterns],
        }, index=common_patterns)
        chi2, p, dof, _ = chi2_contingency(ct_robust)
        print(f"\nChi-squared (full vs AI-tools patterns): chi2={chi2:.2f}, p={p:.4f}, dof={dof}")
        print(f"Cramer's V: {cramers_v(ct_robust):.3f}")

In [ ]:
# RQ3a Summary Stats
print("=" * 70)
print("RQ3a SUMMARY: Sequential Progression")
print("=" * 70)

total = len(seq_df)
for pat in pattern_order:
    n = (seq_df["pattern"] == pat).sum()
    print(f"  {pat}: {n} ({100*n/total:.1f}%)")

print(f"\nTop 3 sequences:")
for seq, cnt in seq_df["sequence"].value_counts().head(3).items():
    print(f"  {seq}: {cnt} repos ({100*cnt/total:.1f}%)")

print(f"\nEntry transitions (L1 \u2192 first level):")
for lvl, n in entry_level.value_counts().sort_index().items():
    print(f"  L1\u2192L{int(lvl)}: {n} ({100*n/len(entry_level):.1f}%)")
_pos = latency[latency > 0]
print(f"  L1 duration: median={latency.median():.0f}d observed (lower bound; "
      f"{len(_pos)} with measurable pre-adoption history, median={_pos.median():.0f}d)")

print(f"\nTransition times (strictly positive gaps only):")
for label, days in transitions.items():
    print(f"  {label}: median={days.median():.0f}d, n={len(days)}")

### RQ3a Supplement: Temporal-Structural Concordance

Does the temporal order in which levels are introduced match the a priori L2 < L3 < L4 hierarchy? This bridges the cross-sectional structure (RQ1) with temporal evidence.

In [ ]:
# --- Analysis 3a.6: Temporal-Structural Concordance ---
# For repos with multi-level temporal data, check whether the temporal
# order of level introduction matches the a priori hierarchy (L2 < L3 < L4).
# This directly tests whether the cross-sectional hierarchy has temporal foundations.

# Build level-intro pivot: repo × level → intro_date
level_pivot = level_intro.pivot_table(
    index="repo_name", columns="level", values="level_intro_date", aggfunc="first"
)

# --- Test 1: Pairwise concordance ---
# For each pair (L2,L3), (L3,L4), (L2,L4): what fraction introduced the lower level first?
pairs = [(2, 3, "L2→L3"), (3, 4, "L3→L4"), (2, 4, "L2→L4")]
print("=== Pairwise Temporal Concordance ===")
print(f"{'Pair':<12s} {'N repos':>8s} {'Concordant':>11s} {'Tied':>6s} {'Discordant':>11s} {'Rate':>8s}")
print("-" * 60)

concordance_results = []
for l_from, l_to, label in pairs:
    if l_from not in level_pivot.columns or l_to not in level_pivot.columns:
        continue
    mask = level_pivot[l_from].notna() & level_pivot[l_to].notna()
    subset = level_pivot.loc[mask]
    n = len(subset)
    if n == 0:
        continue
    
    diff = (subset[l_to] - subset[l_from]).dt.total_seconds()
    concordant = (diff > 0).sum()      # lower level appeared first (correct order)
    tied = (diff == 0).sum()            # same commit/day
    discordant = (diff < 0).sum()       # higher level appeared first (wrong order)
    rate = concordant / n if n > 0 else 0
    
    print(f"{label:<12s} {n:>8d} {concordant:>8d} ({concordant/n*100:4.1f}%) {tied:>4d} {discordant:>8d} ({discordant/n*100:4.1f}%) {rate:>7.1%}")
    concordance_results.append({
        "pair": label, "n": n, 
        "concordant": concordant, "tied": tied, "discordant": discordant,
        "concordance_rate": rate,
    })

# --- Test 2: Full sequence concordance ---
# For repos with 3 levels: what fraction follow exact L2 < L3 < L4 temporal order?
if all(lvl in level_pivot.columns for lvl in (2, 3, 4)):
    three_level = level_pivot.dropna(subset=[2, 3, 4])
    n_three = len(three_level)
else:
    three_level = level_pivot.iloc[0:0]
    n_three = 0
    print("\n=== Full 3-Level Concordance ===\nNo repos with all three levels (no L4-category events under validated attribution)")
if n_three > 0:
    fully_ordered = ((three_level[2] < three_level[3]) & (three_level[3] < three_level[4])).sum()
    l2_first = (three_level[2] <= three_level[3]).sum() & (three_level[2] <= three_level[4]).sum()
    print(f"\n=== Full 3-Level Concordance (repos with L2+L3+L4) ===")
    print(f"Repos with all 3 levels: {n_three}")
    print(f"Fully concordant (L2 < L3 < L4): {fully_ordered} ({fully_ordered/n_three*100:.1f}%)")

# --- Test 3: Category-level temporal ordering ---
# Within each level, which categories tend to be introduced first?
# And across levels, do L2 categories consistently precede L3/L4 categories?
print(f"\n=== Cross-Level Category Temporal Ordering ===")
print("Median introduction date by category (days since earliest event):")

if len(cat_intro) > 0:
    earliest = cat_intro["intro_date"].min()
    cat_intro["days_since_start"] = (cat_intro["intro_date"] - earliest).dt.total_seconds() / 86400
    
    cat_timing = cat_intro.groupby("category")["days_since_start"].agg(["median", "mean", "count"])
    cat_timing["level"] = cat_timing.index.map(CATEGORY_TO_LEVEL_INT)
    cat_timing = cat_timing.sort_values(["level", "median"])
    
    for _, row in cat_timing.iterrows():
        cat = row.name
        print(f"  L{int(row['level'])} {cat:20s}: median={row['median']:7.0f}d, mean={row['mean']:7.0f}d (n={int(row['count'])})")
    
    # Spearman: category level vs median intro timing
    rho, p = spearmanr(cat_timing["level"], cat_timing["median"])
    print(f"\nSpearman(category level, median intro date): rho={rho:.3f}, p={p:.4f}")
    if rho > 0:
        print(f"  Higher-level categories are introduced later (consistent with developmental ordering)")

# --- Visualization ---
conc_df = pd.DataFrame(concordance_results)
if len(conc_df) > 0:
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=conc_df["pair"], 
        y=[r["concordant"]/r["n"]*100 for _, r in conc_df.iterrows()],
        name="Concordant (correct order)",
        marker_color="#22c55e",
    ))
    fig.add_trace(go.Bar(
        x=conc_df["pair"],
        y=[r["tied"]/r["n"]*100 for _, r in conc_df.iterrows()],
        name="Simultaneous",
        marker_color="#94a3b8",
    ))
    fig.add_trace(go.Bar(
        x=conc_df["pair"],
        y=[r["discordant"]/r["n"]*100 for _, r in conc_df.iterrows()],
        name="Discordant (wrong order)",
        marker_color="#ef4444",
    ))
    fig.update_layout(
        title="Temporal-Structural Concordance: Does Level Introduction Follow the A Priori Hierarchy?",
        yaxis_title="% of repos", xaxis_title="Level pair",
        barmode="stack", height=400, width=600,
        legend=dict(orientation="h", yanchor="bottom", y=1.02),
    )
    save_fig(fig, "rq3a_temporal_concordance.png")
    fig.show()

---
## Section 3: RQ3b — Reversals, Plateaus, Abandonment

*How common are non-forward transitions? Where do repos stall or regress?*

Definitions:
- **Reversal**: repo had L3+ artifacts, all now abandoned/deleted
- **Plateau**: no new category introduction for 6+ months
- **Abandonment**: all AI artifacts abandoned/deleted
- **Active growth**: new category intro within last 6 months

In [ ]:
# Classify artifact lifecycles using temporal_health
horizon = pd.Timestamp(timeseries_with_cat["commit_date"].max())
if horizon.tzinfo is None:
    horizon = horizon.tz_localize("UTC")

artifact_lifecycles = []
for (repo, path), grp in timeseries_with_cat.groupby(["repo_name", "artifact_path"]):
    cat = grp["category"].iloc[0]
    lvl = grp["level"].iloc[0]
    lifecycle = classify_artifact_lifecycle(grp, horizon)
    artifact_lifecycles.append({
        "repo_name": repo,
        "artifact_path": path,
        "category": cat,
        "level": lvl,
        "lifecycle": lifecycle,
        "commit_count": len(grp),
        "first_commit": grp["commit_date"].min(),
        "last_commit": grp["commit_date"].max(),
        "n_authors": grp["author_hash"].nunique() if "author_hash" in grp.columns else 1,
    })

lc_df = pd.DataFrame(artifact_lifecycles)
print(f"Artifact lifecycles classified: {len(lc_df)}")
print(f"\nLifecycle distribution:")
print(lc_df["lifecycle"].value_counts().to_string())

In [ ]:
# Classify repos into dynamic states: reversal, plateau, abandonment, active growth
PLATEAU_MONTHS = 6

repo_dynamics = []
for repo, grp in lc_df.groupby("repo_name"):
    repo_cats = cat_intro[cat_intro["repo_name"] == repo]
    last_intro = repo_cats["intro_date"].max() if len(repo_cats) > 0 else None
    
    # Check abandonment: all artifacts abandoned
    all_abandoned = (grp["lifecycle"] == "abandoned").all()
    
    # Check reversal: had L3+ artifacts, now all L3+ are abandoned
    had_l3_plus = (grp["level"] >= 3).any()
    l3_plus_all_abandoned = False
    if had_l3_plus:
        l3_artifacts = grp[grp["level"] >= 3]
        l3_plus_all_abandoned = (l3_artifacts["lifecycle"] == "abandoned").all()
    
    # Check plateau: no new category intro for 6+ months
    months_since_last_intro = None
    is_plateau = False
    if last_intro is not None:
        if last_intro.tzinfo is None:
            last_intro_tz = pd.Timestamp(last_intro, tz="UTC")
        else:
            last_intro_tz = last_intro
        months_since_last_intro = (horizon - last_intro_tz).days / 30.44
        is_plateau = months_since_last_intro >= PLATEAU_MONTHS
    
    # Determine primary state
    if all_abandoned:
        state = "abandoned"
    elif l3_plus_all_abandoned:
        state = "reversal"
    elif is_plateau:
        state = "plateau"
    else:
        state = "active-growth"
    
    current_level = repo_scores.loc[repo_scores["full_repo_name"] == repo, "level"]
    current_level = int(current_level.iloc[0]) if len(current_level) > 0 else None
    
    repo_dynamics.append({
        "repo_name": repo,
        "state": state,
        "current_level": current_level,
        "n_artifacts": len(grp),
        "months_since_last_intro": months_since_last_intro,
        "had_l3_plus": had_l3_plus,
    })

dyn_df = pd.DataFrame(repo_dynamics)

print("Repo dynamic state distribution:")
print(dyn_df["state"].value_counts().to_string())
print(f"\nTotal repos classified: {len(dyn_df)}")

In [ ]:
# Chart 3b.1: Non-forward transition prevalence by level (stacked bar)
state_order = ["active-growth", "plateau", "reversal", "abandoned"]
state_colors = {
    "active-growth": "#22c55e",
    "plateau": "#f59e0b",
    "reversal": "#ef4444",
    "abandoned": "#6b7280",
}

dyn_with_level = dyn_df.dropna(subset=["current_level"])
ct_state = pd.crosstab(dyn_with_level["current_level"], dyn_with_level["state"])

fig = go.Figure()
for state in state_order:
    if state in ct_state.columns:
        fig.add_trace(go.Bar(
            name=state,
            x=[LEVEL_LABELS.get(int(l), f"L{int(l)}") for l in ct_state.index],
            y=ct_state[state],
            marker_color=state_colors[state],
        ))

fig.update_layout(
    title="RQ3b.1: Repo Dynamic State by Current Level",
    barmode="stack",
    xaxis_title="Current Maturity Level",
    yaxis_title="Number of Repos",
    legend_title="State",
    template="plotly_white",
    width=700, height=450,
)
fig.show()
save_fig(fig, "rq3b_state_by_level.png", width=700, height=450)

In [ ]:
# Chart 3b.2: Artifact lifecycle distribution by maturity level (grouped bar)
lc_by_level = pd.crosstab(lc_df["level"], lc_df["lifecycle"])
lc_pct = lc_by_level.div(lc_by_level.sum(axis=1), axis=0) * 100

lifecycle_order = ["steady", "burst", "set-and-forget", "abandoned"]
lifecycle_colors = {"steady": "#22c55e", "burst": "#3b82f6", "set-and-forget": "#f59e0b", "abandoned": "#ef4444"}

fig = go.Figure()
for lc in lifecycle_order:
    if lc in lc_pct.columns:
        fig.add_trace(go.Bar(
            name=lc,
            x=[LEVEL_LABELS.get(int(l), f"L{int(l)}") for l in lc_pct.index],
            y=lc_pct[lc],
            marker_color=lifecycle_colors[lc],
        ))

fig.update_layout(
    title="RQ3b.2: Artifact Lifecycle Distribution by Level",
    barmode="group",
    xaxis_title="Maturity Level",
    yaxis_title="% of Artifacts",
    legend_title="Lifecycle",
    template="plotly_white",
    width=800, height=450,
)
fig.show()
save_fig(fig, "rq3b_lifecycle_by_level.png", width=800, height=450)

In [ ]:
# Chart 3b.3: Artifact deletion analysis (% deleted by level, timing)
# Identify deleted artifacts from timeseries (action == "deleted")
deleted_events = timeseries_with_cat[timeseries_with_cat["action"] == "deleted"].copy()

if len(deleted_events) > 0:
    deleted_by_level = deleted_events.groupby("level")["artifact_path"].nunique()
    total_by_level = timeseries_with_cat.groupby("level")["artifact_path"].nunique()
    del_pct = (deleted_by_level / total_by_level * 100).fillna(0)
    
    fig = go.Figure(go.Bar(
        x=[LEVEL_LABELS.get(int(l), f"L{int(l)}") for l in del_pct.index],
        y=del_pct.values,
        marker_color=[LEVEL_COLORS.get(int(l), "#999") for l in del_pct.index],
    ))
    fig.update_layout(
        title="RQ3b.3: Artifact Deletion Rate by Level",
        xaxis_title="Maturity Level",
        yaxis_title="% of Artifacts Deleted",
        template="plotly_white",
        width=600, height=400,
    )
    fig.show()
    save_fig(fig, "rq3b_deletion_by_level.png", width=600, height=400)
    
    print(f"Deleted artifacts by level:")
    for l in sorted(del_pct.index):
        lvl_lbl = f"L{int(l)}" + ("-category" if int(l) == 4 else "")
        print(f"  {lvl_lbl}: {del_pct[l]:.1f}% ({deleted_by_level.get(l, 0)} / {total_by_level.get(l, 0)})")
else:
    print("No deletion events found in timeseries")

In [ ]:
# Chart 3b.4: Plateau duration distribution (histogram by stalled level)
plateau_repos = dyn_df[dyn_df["state"] == "plateau"].dropna(subset=["current_level", "months_since_last_intro"])

if len(plateau_repos) > 0:
    fig = go.Figure()
    for lvl in sorted(plateau_repos["current_level"].unique()):
        lvl_data = plateau_repos[plateau_repos["current_level"] == lvl]["months_since_last_intro"]
        fig.add_trace(go.Histogram(
            x=lvl_data,
            name=LEVEL_LABELS.get(int(lvl), f"L{int(lvl)}"),
            marker_color=LEVEL_COLORS.get(int(lvl), "#999"),
            opacity=0.7,
            nbinsx=15,
        ))
    
    fig.update_layout(
        title="RQ3b.4: Plateau Duration by Stalled Level",
        xaxis_title="Months Since Last Category Introduction",
        yaxis_title="Number of Repos",
        barmode="overlay",
        template="plotly_white",
        width=700, height=400,
    )
    fig.show()
    save_fig(fig, "rq3b_plateau_duration.png", width=700, height=400)
    
    print(f"Plateau repos: {len(plateau_repos)}")
    print(f"Median plateau duration: {plateau_repos['months_since_last_intro'].median():.1f} months")
else:
    print("No plateau repos found")

In [ ]:
# Chi-squared: are plateau/reversal rates independent of level? + Cramer's V
dyn_with_level_clean = dyn_with_level.dropna(subset=["current_level"])
ct_chi = pd.crosstab(dyn_with_level_clean["current_level"], dyn_with_level_clean["state"])

if ct_chi.shape[0] >= 2 and ct_chi.shape[1] >= 2:
    chi2, p, dof, expected = chi2_contingency(ct_chi)
    v = cramers_v(ct_chi)
    print(f"Chi-squared (state × level): chi2={chi2:.2f}, p={p:.4f}, dof={dof}")
    print(f"Cramer's V: {v:.3f}")
    print(f"\nContingency table:")
    print(ct_chi.to_string())
else:
    print("Insufficient categories for chi-squared test")

In [ ]:
# Robustness: AI-tools subset
ai_dyn = dyn_df[dyn_df["repo_name"].isin(ai_tool_repos)]

print("=== AI-Tools Subset Robustness (RQ3b) ===")
print(f"Repos: {len(ai_dyn)}")
print(f"\nState distribution (AI-tools):")
print(ai_dyn["state"].value_counts().to_string())

# Compare proportions
if len(ai_dyn) >= 10:
    all_states = dyn_df["state"].value_counts()
    ai_states = ai_dyn["state"].value_counts()
    common = sorted(set(all_states.index) & set(ai_states.index))
    if len(common) >= 2:
        ct_r = pd.DataFrame({
            "full": [all_states.get(s, 0) for s in common],
            "ai_tools": [ai_states.get(s, 0) for s in common],
        }, index=common)
        chi2, p, dof, _ = chi2_contingency(ct_r)
        print(f"\nChi-squared (full vs AI-tools states): chi2={chi2:.2f}, p={p:.4f}")
        print(f"Cramer's V: {cramers_v(ct_r):.3f}")

In [ ]:
# RQ3b Summary Stats
print("=" * 70)
print("RQ3b SUMMARY: Reversals, Plateaus, Abandonment")
print("=" * 70)

total_dyn = len(dyn_df)
for state in state_order:
    n = (dyn_df["state"] == state).sum()
    print(f"  {state}: {n} ({100*n/total_dyn:.1f}%)")

print(f"\nLifecycle distribution across {len(lc_df)} artifacts:")
for lc in lifecycle_order:
    n = (lc_df["lifecycle"] == lc).sum()
    print(f"  {lc}: {n} ({100*n/len(lc_df):.1f}%)")

---
## Section 4: RQ3c — Maintenance Effort

*Does maintenance effort scale with maturity level?*

Metrics:
- `commits_per_month` touching AI artifacts
- `maintenance_ratio` = active months / total months
- `unique_authors` contributing to AI artifacts

In [ ]:
# Compute maintenance metrics per repo from timeseries
repo_maintenance = []
for repo, grp in timeseries_with_cat.groupby("repo_name"):
    first = grp["commit_date"].min()
    last = grp["commit_date"].max()
    span_days = max((last - first).total_seconds() / 86400, 1)
    span_months = span_days / 30.44
    
    total_commits = len(grp)
    commits_per_month = total_commits / max(span_months, 1)
    
    # Active months: months with at least one commit
    grp_monthly = grp.set_index("commit_date").resample("MS").size()
    active_months = (grp_monthly > 0).sum()
    total_month_span = max(len(grp_monthly), 1)
    maintenance_ratio = active_months / total_month_span
    
    unique_authors = grp["author_hash"].nunique() if "author_hash" in grp.columns else 1
    
    current_level = repo_scores.loc[repo_scores["full_repo_name"] == repo, "level"]
    current_level = int(current_level.iloc[0]) if len(current_level) > 0 else None
    
    n_artifacts = grp["artifact_path"].nunique()
    
    repo_maintenance.append({
        "repo_name": repo,
        "current_level": current_level,
        "total_commits": total_commits,
        "commits_per_month": commits_per_month,
        "maintenance_ratio": maintenance_ratio,
        "unique_authors": unique_authors,
        "active_months": active_months,
        "span_months": span_months,
        "n_artifacts": n_artifacts,
    })

maint_df = pd.DataFrame(repo_maintenance)
maint_df = maint_df.dropna(subset=["current_level"])
maint_df["current_level"] = maint_df["current_level"].astype(int)

print(f"Repos with maintenance metrics: {len(maint_df)}")
print(f"\nMedian commits/month by level:")
for lvl in sorted(maint_df["current_level"].unique()):
    subset = maint_df[maint_df["current_level"] == lvl]
    print(f"  L{lvl}: {subset['commits_per_month'].median():.2f} (n={len(subset)})")

In [ ]:
# Chart 3c.1: Commits/month by level (box plot + Kruskal-Wallis)
fig = go.Figure()
groups_cpm = []
for lvl in sorted(maint_df["current_level"].unique()):
    subset = maint_df[maint_df["current_level"] == lvl]["commits_per_month"]
    groups_cpm.append(subset)
    fig.add_trace(go.Box(
        y=subset,
        name=LEVEL_LABELS.get(lvl, f"L{lvl}"),
        marker_color=LEVEL_COLORS.get(lvl, "#999"),
        boxmean="sd",
    ))

if len(groups_cpm) >= 3:
    h_stat, p_val = kruskal(*groups_cpm)
    print(f"Kruskal-Wallis (commits/month × level): H={h_stat:.2f}, p={p_val:.4f}")

fig.update_layout(
    title="RQ3c.1: AI Artifact Commits per Month by Level",
    yaxis_title="Commits / Month",
    template="plotly_white",
    showlegend=False,
    width=700, height=450,
)
fig.show()
save_fig(fig, "rq3c_commits_per_month.png", width=700, height=450)

In [ ]:
# Chart 3c.2: Maintenance ratio by level (box plot + Spearman)
fig = go.Figure()
for lvl in sorted(maint_df["current_level"].unique()):
    subset = maint_df[maint_df["current_level"] == lvl]["maintenance_ratio"]
    fig.add_trace(go.Box(
        y=subset,
        name=LEVEL_LABELS.get(lvl, f"L{lvl}"),
        marker_color=LEVEL_COLORS.get(lvl, "#999"),
        boxmean="sd",
    ))

rho, p = spearmanr(maint_df["current_level"], maint_df["maintenance_ratio"])
print(f"Spearman (level vs maintenance_ratio): rho={rho:.3f}, p={p:.4f}")

fig.update_layout(
    title=f"RQ3c.2: Maintenance Ratio by Level (Spearman rho={rho:.3f})",
    yaxis_title="Maintenance Ratio (active months / total months)",
    template="plotly_white",
    showlegend=False,
    width=700, height=450,
)
fig.show()
save_fig(fig, "rq3c_maintenance_ratio.png", width=700, height=450)

In [ ]:
# Chart 3c.3: Lifecycle composition by level (stacked 100% bar)
# Per-repo dominant lifecycle
repo_lc_summary = lc_df.groupby(["repo_name", "lifecycle"]).size().reset_index(name="count")
repo_dominant = repo_lc_summary.loc[repo_lc_summary.groupby("repo_name")["count"].idxmax()]
repo_dominant = repo_dominant.merge(
    maint_df[["repo_name", "current_level"]],
    on="repo_name",
    how="inner"
)

lc_by_level_repo = pd.crosstab(repo_dominant["current_level"], repo_dominant["lifecycle"])
lc_pct_repo = lc_by_level_repo.div(lc_by_level_repo.sum(axis=1), axis=0) * 100

fig = go.Figure()
for lc in lifecycle_order:
    if lc in lc_pct_repo.columns:
        fig.add_trace(go.Bar(
            name=lc,
            x=[LEVEL_LABELS.get(int(l), f"L{int(l)}") for l in lc_pct_repo.index],
            y=lc_pct_repo[lc],
            marker_color=lifecycle_colors.get(lc, "#999"),
        ))

fig.update_layout(
    title="RQ3c.3: Dominant Lifecycle by Level (% of Repos)",
    barmode="stack",
    xaxis_title="Maturity Level",
    yaxis_title="% of Repos",
    template="plotly_white",
    width=700, height=450,
)
fig.show()
save_fig(fig, "rq3c_lifecycle_composition.png", width=700, height=450)

In [ ]:
# Chart 3c.4: Author diversity by level (box plot + Spearman)
fig = go.Figure()
for lvl in sorted(maint_df["current_level"].unique()):
    subset = maint_df[maint_df["current_level"] == lvl]["unique_authors"]
    fig.add_trace(go.Box(
        y=subset,
        name=LEVEL_LABELS.get(lvl, f"L{lvl}"),
        marker_color=LEVEL_COLORS.get(lvl, "#999"),
        boxmean="sd",
    ))

rho_auth, p_auth = spearmanr(maint_df["current_level"], maint_df["unique_authors"])
print(f"Spearman (level vs unique_authors): rho={rho_auth:.3f}, p={p_auth:.4f}")

fig.update_layout(
    title=f"RQ3c.4: Unique Authors on AI Artifacts by Level (Spearman rho={rho_auth:.3f})",
    yaxis_title="Unique Authors",
    template="plotly_white",
    showlegend=False,
    width=700, height=450,
)
fig.show()
save_fig(fig, "rq3c_author_diversity.png", width=700, height=450)

In [ ]:
# Chart 3c.5: Maintenance trajectory — rolling 3-month commit rate for repos with 12+ months history
long_repos = maint_df[maint_df["span_months"] >= 12]["repo_name"].tolist()
print(f"Repos with 12+ months history: {len(long_repos)}")

if len(long_repos) > 0:
    # Compute rolling 3-month commit counts per level
    trajectories = {}
    for repo in long_repos:
        grp = timeseries_with_cat[timeseries_with_cat["repo_name"] == repo].copy()
        lvl = maint_df.loc[maint_df["repo_name"] == repo, "current_level"].iloc[0]
        
        monthly = grp.set_index("commit_date").resample("MS").size()
        rolling = monthly.rolling(3, min_periods=1).mean()
        
        # Normalize to months since first commit
        if len(rolling) > 0:
            start = rolling.index[0]
            months_offset = [(d - start).days / 30.44 for d in rolling.index]
            trajectories.setdefault(lvl, []).append((months_offset, rolling.values))
    
    fig = go.Figure()
    for lvl in sorted(trajectories.keys()):
        # Aggregate: compute median trajectory
        max_len = max(len(t[0]) for t in trajectories[lvl])
        all_vals = np.full((len(trajectories[lvl]), max_len), np.nan)
        for i, (months, vals) in enumerate(trajectories[lvl]):
            all_vals[i, :len(vals)] = vals
        
        median_vals = np.nanmedian(all_vals, axis=0)
        x_months = np.arange(max_len)
        # Trim to where we have at least 5 repos
        valid = np.sum(~np.isnan(all_vals), axis=0)
        mask = valid >= min(5, len(trajectories[lvl]))
        if mask.any():
            last_valid = np.where(mask)[0][-1]
            fig.add_trace(go.Scatter(
                x=x_months[:last_valid+1],
                y=median_vals[:last_valid+1],
                name=f"{LEVEL_LABELS.get(lvl, f'L{lvl}')} (n={len(trajectories[lvl])})",
                line=dict(color=LEVEL_COLORS.get(lvl, "#999"), width=2),
            ))
    
    fig.update_layout(
        title="RQ3c.5: Median Maintenance Trajectory (3-month rolling commits)",
        xaxis_title="Months Since First AI Artifact",
        yaxis_title="Rolling 3-month Avg Commits",
        template="plotly_white",
        width=800, height=450,
    )
    fig.show()
    save_fig(fig, "rq3c_maintenance_trajectory.png", width=800, height=450)
else:
    print("No repos with 12+ months history")

In [ ]:
# Chart 3c.6: Category × lifecycle heatmap
cat_lc = pd.crosstab(lc_df["category"], lc_df["lifecycle"])
cat_lc_pct = cat_lc.div(cat_lc.sum(axis=1), axis=0) * 100

# Order categories by level then name
cat_order = sorted(CATEGORY_TO_LEVEL_INT.keys(), key=lambda c: (CATEGORY_TO_LEVEL_INT[c], c))
lc_order_display = ["steady", "burst", "set-and-forget", "abandoned"]

# Filter to available categories and lifecycles
cat_order = [c for c in cat_order if c in cat_lc_pct.index]
lc_order_display = [l for l in lc_order_display if l in cat_lc_pct.columns]

z = cat_lc_pct.loc[cat_order, lc_order_display].values

fig = go.Figure(go.Heatmap(
    z=z,
    x=lc_order_display,
    y=[f"{c} (L{CATEGORY_TO_LEVEL_INT[c]})" for c in cat_order],
    colorscale="YlOrRd",
    text=np.round(z, 1),
    texttemplate="%{text}%",
    textfont={"size": 11},
    colorbar_title="%",
))

fig.update_layout(
    title="RQ3c.6: Category × Lifecycle Heatmap",
    template="plotly_white",
    width=700, height=500,
    margin=dict(l=150),
)
fig.show()
save_fig(fig, "rq3c_category_lifecycle_heatmap.png", width=700, height=500)

In [ ]:
# Partial Spearman: level vs maintenance_ratio controlling for artifact_count
# Same confound-control pattern as RQ2c
from scipy.stats import rankdata

def partial_spearman(x, y, z):
    """Partial Spearman correlation between x and y controlling for z."""
    rx = rankdata(x)
    ry = rankdata(y)
    rz = rankdata(z)
    
    # Residualize x and y on z
    def residualize(a, b):
        rho_ab, _ = spearmanr(a, b)
        return a - rho_ab * b
    
    # Partial correlation formula
    rho_xy, _ = spearmanr(rx, ry)
    rho_xz, _ = spearmanr(rx, rz)
    rho_yz, _ = spearmanr(ry, rz)
    
    numerator = rho_xy - rho_xz * rho_yz
    denominator = np.sqrt((1 - rho_xz**2) * (1 - rho_yz**2))
    
    if denominator == 0:
        return 0.0, 1.0
    
    partial_rho = numerator / denominator
    
    # Approximate p-value using t-distribution
    from scipy.stats import t as t_dist
    n = len(x)
    t_stat = partial_rho * np.sqrt((n - 3) / (1 - partial_rho**2))
    p_val = 2 * (1 - t_dist.cdf(abs(t_stat), df=n - 3))
    
    return partial_rho, p_val

# Raw Spearman
rho_raw, p_raw = spearmanr(maint_df["current_level"], maint_df["maintenance_ratio"])
print(f"Raw Spearman (level vs maintenance_ratio): rho={rho_raw:.3f}, p={p_raw:.4f}")

# Partial Spearman controlling for artifact_count
rho_partial, p_partial = partial_spearman(
    maint_df["current_level"].values,
    maint_df["maintenance_ratio"].values,
    maint_df["n_artifacts"].values,
)
print(f"Partial Spearman (controlling artifact_count): rho={rho_partial:.3f}, p={p_partial:.4f}")

# Also for commits_per_month
rho_cpm, p_cpm = spearmanr(maint_df["current_level"], maint_df["commits_per_month"])
rho_cpm_partial, p_cpm_partial = partial_spearman(
    maint_df["current_level"].values,
    maint_df["commits_per_month"].values,
    maint_df["n_artifacts"].values,
)
print(f"\nRaw Spearman (level vs commits/month): rho={rho_cpm:.3f}, p={p_cpm:.4f}")
print(f"Partial Spearman (controlling artifact_count): rho={rho_cpm_partial:.3f}, p={p_cpm_partial:.4f}")

In [ ]:
# Robustness: AI-tools subset (RQ3c)
ai_maint = maint_df[maint_df["repo_name"].isin(ai_tool_repos)]

print("=== AI-Tools Subset Robustness (RQ3c) ===")
print(f"Repos: {len(ai_maint)}")

if len(ai_maint) >= 10:
    rho_ai, p_ai = spearmanr(ai_maint["current_level"], ai_maint["maintenance_ratio"])
    print(f"Spearman (level vs maintenance_ratio, AI-tools): rho={rho_ai:.3f}, p={p_ai:.4f}")
    
    rho_ai_cpm, p_ai_cpm = spearmanr(ai_maint["current_level"], ai_maint["commits_per_month"])
    print(f"Spearman (level vs commits/month, AI-tools): rho={rho_ai_cpm:.3f}, p={p_ai_cpm:.4f}")
    
    rho_ai_auth, p_ai_auth = spearmanr(ai_maint["current_level"], ai_maint["unique_authors"])
    print(f"Spearman (level vs unique_authors, AI-tools): rho={rho_ai_auth:.3f}, p={p_ai_auth:.4f}")
    
    # Kruskal-Wallis on AI-tools subset
    groups_ai = [ai_maint[ai_maint["current_level"] == l]["commits_per_month"] for l in sorted(ai_maint["current_level"].unique())]
    groups_ai = [g for g in groups_ai if len(g) >= 2]
    if len(groups_ai) >= 3:
        h, p = kruskal(*groups_ai)
        print(f"Kruskal-Wallis (commits/month × level, AI-tools): H={h:.2f}, p={p:.4f}")

In [ ]:
# RQ3c Summary Stats
print("=" * 70)
print("RQ3c SUMMARY: Maintenance Effort")
print("=" * 70)

print(f"\nRepos analyzed: {len(maint_df)}")
print(f"\nCommits/month by level:")
for lvl in sorted(maint_df["current_level"].unique()):
    s = maint_df[maint_df["current_level"] == lvl]["commits_per_month"]
    print(f"  L{lvl}: median={s.median():.2f}, mean={s.mean():.2f}, n={len(s)}")

print(f"\nMaintenance ratio by level:")
for lvl in sorted(maint_df["current_level"].unique()):
    s = maint_df[maint_df["current_level"] == lvl]["maintenance_ratio"]
    print(f"  L{lvl}: median={s.median():.2f}, mean={s.mean():.2f}")

print(f"\nUnique authors by level:")
for lvl in sorted(maint_df["current_level"].unique()):
    s = maint_df[maint_df["current_level"] == lvl]["unique_authors"]
    print(f"  L{lvl}: median={s.median():.1f}, mean={s.mean():.1f}")

---
## Section 5: Consolidated Summary

In [ ]:
# Consolidated RQ3 summary
print("=" * 70)
print("RQ3 CONSOLIDATED SUMMARY: Temporal Maturity Dynamics")
print("=" * 70)

print(f"\nDataset: {timeseries_with_cat['repo_name'].nunique()} repos with categorized timeseries")
print(f"Total events: {len(timeseries_with_cat):,}")
print(f"Date range: {timeseries_with_cat['commit_date'].min().strftime('%Y-%m-%d')} — {timeseries_with_cat['commit_date'].max().strftime('%Y-%m-%d')}")

print(f"\n--- RQ3a: Sequential Progression ---")
total = len(seq_df)
sequential_n = (seq_df["pattern"] == "sequential").sum()
print(f"  Sequential progression: {sequential_n}/{total} ({100*sequential_n/total:.1f}%)")
print(f"  Non-sequential: {(seq_df['pattern'] == 'non-sequential').sum()}/{total}")
for label, days in transitions.items():
    print(f"  {label} median time: {days.median():.0f} days (n={len(days)})")

print(f"\n--- RQ3b: Reversals & Plateaus ---")
total_dyn = len(dyn_df)
for state in state_order:
    n = (dyn_df["state"] == state).sum()
    print(f"  {state}: {n} ({100*n/total_dyn:.1f}%)")

print(f"\n--- RQ3c: Maintenance Effort ---")
print(f"  Spearman (level vs maintenance_ratio): rho={rho_raw:.3f}, p={p_raw:.4f}")
print(f"  Partial Spearman (controlling artifact_count): rho={rho_partial:.3f}, p={p_partial:.4f}")
print(f"  Spearman (level vs unique_authors): rho={rho_auth:.3f}, p={p_auth:.4f}")

print(f"\n{'=' * 70}")

In [ ]:
# Summary dashboard figure
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[
        "Progression Patterns", "Top Sequences", "Transition Times",
        "Dynamic States", "Lifecycle by Level", "Maintenance by Level",
    ],
    specs=[
        [{"type": "domain"}, {"type": "xy"}, {"type": "xy"}],
        [{"type": "domain"}, {"type": "xy"}, {"type": "xy"}],
    ],
    horizontal_spacing=0.1,
    vertical_spacing=0.15,
)

# 1. Progression patterns (pie)
pat_counts = seq_df["pattern"].value_counts()
fig.add_trace(go.Pie(
    labels=pat_counts.index,
    values=pat_counts.values,
    marker_colors=[pattern_colors.get(p, "#999") for p in pat_counts.index],
    textinfo="percent",
    showlegend=False,
), row=1, col=1)

# 2. Top sequences (horizontal bar)
top5 = seq_df["sequence"].value_counts().head(5)
fig.add_trace(go.Bar(
    y=top5.index[::-1],
    x=top5.values[::-1],
    orientation="h",
    marker_color="#3b82f6",
    showlegend=False,
), row=1, col=2)

# 3. Transition times (box)
for label, color in [("L2→L3", LEVEL_COLORS[3]), ("L3→L4", LEVEL_COLORS[4])]:
    if label in transitions:
        fig.add_trace(go.Box(
            y=transitions[label],
            name=label,
            marker_color=color,
            showlegend=False,
        ), row=1, col=3)

# 4. Dynamic states (pie)
st_counts = dyn_df["state"].value_counts()
fig.add_trace(go.Pie(
    labels=st_counts.index,
    values=st_counts.values,
    marker_colors=[state_colors.get(s, "#999") for s in st_counts.index],
    textinfo="percent",
    showlegend=False,
), row=2, col=1)

# 5. Lifecycle by level (stacked bar)
for lc in lifecycle_order:
    if lc in lc_pct.columns:
        fig.add_trace(go.Bar(
            x=[f"L{int(l)}" for l in lc_pct.index],
            y=lc_pct[lc],
            name=lc,
            marker_color=lifecycle_colors.get(lc, "#999"),
            showlegend=False,
        ), row=2, col=2)

# 6. Maintenance by level (box)
for lvl in sorted(maint_df["current_level"].unique()):
    subset = maint_df[maint_df["current_level"] == lvl]["commits_per_month"]
    fig.add_trace(go.Box(
        y=subset,
        name=f"L{lvl}",
        marker_color=LEVEL_COLORS.get(lvl, "#999"),
        showlegend=False,
    ), row=2, col=3)

fig.update_layout(
    title="RQ3: Temporal Maturity Dynamics — Summary Dashboard<br><sub>categories from the validated notebook-13 attribution (rq1_file_predictions.parquet)</sub>",
    template="plotly_white",
    height=800,
    width=1200,
    barmode="stack",
)
fig.show()
save_fig(fig, "rq3_dynamics_summary.png", width=1200, height=800)